# Master Dataset Pipeline (2021–2024)

Assembles market transactions and fundamental energy time series for Germany and the four TSO control areas (DE1–DE4) onto a continuous 15-minute UTC grid.

In [ ]:
import os
import sys
import glob
import time
import pandas as pd
import numpy as np

# Project Paths
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "Data")
FEATURES_CSV = os.path.join(BASE_DIR, "features.csv")

TSO_TO_DELIVERY_AREA = {
    "TransnetBW": "DE1",
    "Amprion": "DE2",
    "TenneT": "DE3",
    "TenneT TSO": "DE3",
    "50Hertz": "DE4",
    "Deutschland": "DE",
    "Germany": "DE",
}

DELIVERY_AREAS = list(dict.fromkeys(TSO_TO_DELIVERY_AREA.values()))
REGIONAL_AREAS = [area for area in DELIVERY_AREAS if area != "DE"]

def get_tso_zone(text):
    """Maps TSO name found in a string (column name or filename) to DE1-DE4 or DE."""
    for tso, zone in TSO_TO_DELIVERY_AREA.items():
        if tso in text:
            return zone
    return None

# Date Spine
# First delivery: 2021-10-01 00:00 local (22:00 UTC). The rows before it only absorb the longest ex-post lag.
sys.path.insert(0, BASE_DIR)
from config import LAG_QH_BALANCING, LAG_QH_CROSS_BORDER, LAG_QH_ENTSOE

FIRST_DELIVERY_UTC = pd.Timestamp("2021-09-30 22:00:00")
START_UTC = str(FIRST_DELIVERY_UTC - pd.Timedelta(minutes=15 * max(LAG_QH_BALANCING, LAG_QH_ENTSOE, LAG_QH_CROSS_BORDER)))
END_UTC = "2024-12-31 22:45:00"

BUFFER_START_UTC = str(pd.Timestamp(START_UTC) - pd.Timedelta(days=7)) # 7-day look-ahead buffer
FREQ = "15min"

print(f"Base Directory:   {BASE_DIR}")
print(f"Delivery Horizon:  {START_UTC} to {END_UTC}")

## 1. Date Spine
Build a 15-minute UTC time grid from 2021-09-23 to 2024-12-31, including a 7-day lookback buffer.

In [ ]:
# 1. Date Spine Creation (with 7-day lag buffer)
date_range = pd.date_range(start=BUFFER_START_UTC, end=END_UTC, freq=FREQ, tz="UTC")
master_df = pd.DataFrame(date_range, columns=["Date"])

print(f"Created UTC date spine with {len(master_df):,} qh (incl. 7-day lag buffer).")
print(f"Spine Start: {master_df['Date'].iloc[0]}")
print(f"Spine End:   {master_df['Date'].iloc[-1]}")
master_df.head(3)


## 2. Balancing Energy
Load operational aFRR (SRL) and mFRR (MRL) activations from netztransparenz.org (2021–2024) and map TSO control areas to DE1–DE4.

In [ ]:
# 2. Load Balancing Energy (SRL / aFRR & MRL / mFRR)
be_dir = os.path.join(DATA_DIR, "Balancing Energy")
res_dfs = []

for res_type in ["SRL", "MRL"]:
    files = sorted(glob.glob(os.path.join(be_dir, res_type, "*.csv")))
    type_dfs = []
    for file_path in files:
        df = pd.read_csv(file_path, sep=";", decimal=",", na_values=["N.A."])
        date_series = pd.to_datetime(df["Datum"] + " " + df["von"], format="%d.%m.%Y %H:%M", utc=True)
        df = df.drop(columns=["Datum", "Zeitzone", "von", "bis", "Einheit", "MOL-Abweichung"], errors="ignore")

        rename_map = {}
        for col in df.columns:
            zone = get_tso_zone(col)
            sign = "positive" if "Positiv" in col else "negative"
            rename_map[col] = f"{zone}_{res_type}_{sign}"

        df = df.rename(columns=rename_map)
        df["Date"] = date_series
        type_dfs.append(df)

    combined_type = pd.concat(type_dfs, ignore_index=True)
    res_dfs.append(combined_type)

be_df = res_dfs[0]
for other in res_dfs[1:]:
    be_df = be_df.merge(other, on="Date", how="outer")

# Merge balancing reserves (unactivated/missing intervals filled strictly with 0.0 MW)
master_df = master_df.merge(be_df.fillna(0.0), on="Date", how="left")

print(f"Merged Balancing Energy: {len(be_df.columns) - 1} reserve activation features.")
master_df.filter(regex=r"SRL|MRL").head(3)

## 3. Fundamentals
Load total load, solar, wind (onshore/offshore), and cross-border commercial flows from Fraunhofer ISE Energy-Charts (ENTSO-E data).

In [ ]:
# 3. Load Fundamentals (Load, Solar, Wind Onshore & Offshore, Cross-Border Flows)
fund_base = os.path.join(DATA_DIR, "Fundamentals")
categories = [
    ("LOAD", "load"),
    ("SOLAR", "solar"),
    ("ONSHORE", "wind_onshore"),
    ("OFFSHORE", "wind_offshore")
]

all_cat_dfs = []
for folder_name, prefix in categories:
    cat_dir = os.path.join(fund_base, folder_name)
    files = sorted(glob.glob(os.path.join(cat_dir, "*.csv")))

    zone_files = {}
    for f in files:
        zone = get_tso_zone(os.path.basename(f))
        if zone:
            if zone not in zone_files:
                zone_files[zone] = []
            zone_files[zone].append(f)

    zone_dfs = []
    for zone, z_files in zone_files.items():
        sub_dfs = []
        for f in z_files:
            df = pd.read_csv(f, skiprows=[1], parse_dates=["Date (UTC)"])
            df.columns = ["Date", f"{zone}_{prefix}_actual", f"{zone}_{prefix}_forecast"]
            sub_dfs.append(df)

        z_df = pd.concat(sub_dfs, ignore_index=True)
        act_c = f"{zone}_{prefix}_actual"
        fc_c = f"{zone}_{prefix}_forecast"
        z_df[f"{zone}_{prefix}_diff"] = z_df[act_c] - z_df[fc_c]
        zone_dfs.append(z_df)

    combined_cat = zone_dfs[0]
    for other in zone_dfs[1:]:
        combined_cat = combined_cat.merge(other, on="Date", how="outer")
    all_cat_dfs.append(combined_cat)

# Load Cross-Border Flows (Germany)
cb_dir = os.path.join(fund_base, "CROSS_BORDER")
if os.path.isdir(cb_dir):
    cb_files = sorted(glob.glob(os.path.join(cb_dir, "*.csv")))
    cb_dfs = []
    for f in cb_files:
        df = pd.read_csv(f, skiprows=[1], parse_dates=["Date (UTC)"])
        df.columns = ["Date", "DE_cross_border_trading"]
        cb_dfs.append(df)
    if cb_dfs:
        combined_cb = pd.concat(cb_dfs, ignore_index=True)
        all_cat_dfs.append(combined_cb)

fund_df = all_cat_dfs[0]
for other in all_cat_dfs[1:]:
    fund_df = fund_df.merge(other, on="Date", how="outer")

master_df = master_df.merge(fund_df, on="Date", how="left")
print(f"Merged Fundamentals: {len(fund_df.columns) - 1} generation, demand, forecast & cross-border features.")
master_df.filter(regex=r"load|solar|wind|cross_border").head(3)

## 4. Auction Spot Prices
Load Day-Ahead hourly auction prices (12:00 D-1) and Intraday 15-minute auction prices (15:00 D-1).

In [ ]:
# 4. Load Auction Spot Prices (DA and ID)
def parse_epex_auction(files, value_name):
    dfs = []
    for f in files:
        df = pd.read_csv(f, comment="#")
        period_cols = [c for c in df.columns if c.startswith("Hour ")]
        parts = [
            pd.DataFrame({"Delivery day": df["Delivery day"], "Period": col, value_name: df[col]})
            for col in period_cols
        ]
        dfs.append(pd.concat(parts, ignore_index=True).dropna(subset=[value_name]))

    combined = pd.concat(dfs, ignore_index=True)
    dates = pd.to_datetime(combined["Delivery day"], dayfirst=True)

    if "Q" in combined["Period"].iloc[0]:
        parsed = combined["Period"].str.extract(r"Hour\s+(\d+)([AB])?\s+Q([1-4])")
        hours = parsed[0].astype(int) - 1
        quarters = parsed[2].astype(int) - 1
        combined["local_time"] = dates + pd.to_timedelta(hours, unit="h") + pd.to_timedelta(quarters * 15, unit="m")
        expanded = combined
    else:
        hours = combined["Period"].str.extract(r"Hour\s+(\d+)([AB])?")[0].astype(int) - 1
        combined["local_time"] = dates + pd.to_timedelta(hours, unit="h")
        expanded = pd.concat([combined.assign(local_time=combined["local_time"] + pd.Timedelta(minutes=m)) for m in [0, 15, 30, 45]], ignore_index=True)

    is_dst = ~expanded["Period"].str.contains("3B")
    local_dt = expanded["local_time"].dt.tz_localize("Europe/Berlin", ambiguous=is_dst, nonexistent="shift_forward")
    expanded["Date"] = local_dt.dt.tz_convert("UTC")

    return expanded[["Date", value_name]].sort_values("Date").drop_duplicates(subset=["Date"])

da_files = sorted(glob.glob(os.path.join(DATA_DIR, "DA Auction Spot Prices", "auction_spot_prices_germany_luxembourg_*.csv")))
id_files = sorted(glob.glob(os.path.join(DATA_DIR, "ID Auction Spot Prices", "intraday_auction_spot_prices_15-call_*.csv"))) + \
           sorted(glob.glob(os.path.join(DATA_DIR, "ID Auction Spot Prices", "pan-european_prices_*_IDA1_*.csv")))

da_df = parse_epex_auction(da_files, "Dayahead_Auction_hourly")
id_df = parse_epex_auction(id_files, "Intraday Auction Price")

master_df = master_df.merge(da_df, on="Date", how="left").merge(id_df, on="Date", how="left")
print("Merged Auction Spot Prices: Dayahead_Auction_hourly & Intraday Auction Price.")
master_df[["Date", "Dayahead_Auction_hourly", "Intraday Auction Price"]].head(3)

## 5. EPEX Intraday Continuous Transactions
Load 15-minute rolling VWAP windows, traded volumes, trade counts, and continuous commercial flows for DE1–DE4 and DE.

In [ ]:
# 5. Load EPEX Intraday Continuous Transactions
cont_dir = os.path.join(DATA_DIR, "EPEX Intraday Continuous")

for area in DELIVERY_AREAS:
    files = sorted(glob.glob(os.path.join(cont_dir, f"ID_DelArea_{area}_*.csv")))
    df = pd.concat([pd.read_csv(f, parse_dates=["DeliveryStart"]) for f in files], ignore_index=True)
    df = df.rename(columns={"DeliveryStart": "Date"})

    if area == "DE":
        cols = [c for c in df.columns if c.startswith("VWAP_") 
        and not any(x in c for x in ["total_volume", "total_trades", "upper", "lower"])]
        df = df[["Date"] + cols]
    else:
        cols = [c for c in df.columns if c != "Date" 
        and not any(x in c for x in ["upper", "lower"]) 
        and not c.startswith(f"id_balance_to_{area}_")]
        df = df[["Date"] + cols].rename(columns={c: f"{area}_{c}" for c in cols})

    master_df = master_df.merge(df, on="Date", how="left")

print("Merged EPEX Intraday Continuous metrics.")

## 6. Calendar Indicators
Generate weekday, hourly, and quarter-hour delivery dummies based on Europe/Berlin local time.

In [ ]:
# 6. Add Calendar Indicators (Weekdays only; Hour/Quarter omitted for per-QH models)
local_date = master_df["Date"].dt.tz_convert("Europe/Berlin")
weekday = local_date.dt.dayofweek + 1

dummies = {}
for i in range(1, 8):
    dummies[f"Weekday_{i}"] = (weekday == i).astype(int)

dummy_df = pd.DataFrame(dummies, index=master_df.index)
master_df = pd.concat([master_df, dummy_df], axis=1)

print(f"Generated {len(dummy_df.columns)} calendar dummy indicators.")
master_df.filter(regex=r"Weekday_").head(3)

## 7. Missing Value Imputation
Impute illiquid continuous trading windows: national trajectory uses intraday auction fallback and horizontal forward fill; regional windows (DE1–DE4) fall back to the contemporaneous national VWAP. Volume and flow metrics are filled with zero.

In [ ]:
# 7. Hierarchical Imputation of Continuous Trading Trajectories (Variante B)
windows_15m = [
    "345to360", "330to345", "315to330", "300to315",
    "285to300", "270to285", "255to270", "240to255",
    "225to240", "210to225", "195to210", "180to195",
    "165to180", "150to165", "135to150", "120to135",
    "105to120", "90to105", "75to90", "60to75",
    "45to60", "30to45", "15to30", "0to15"
]

# 1. National DE Trajectory
if "VWAP_345to360" in master_df.columns:
    master_df["VWAP_345to360"] = master_df["VWAP_345to360"].fillna(master_df["Intraday Auction Price"])
    master_df["VWAP_345to360"] = master_df["VWAP_345to360"].fillna(master_df["Dayahead_Auction_hourly"])

for i in range(1, len(windows_15m)):
    curr_w = windows_15m[i]
    prev_w = windows_15m[i - 1]
    curr_col = f"VWAP_{curr_w}"
    prev_col = f"VWAP_{prev_w}"
    if curr_col in master_df.columns and prev_col in master_df.columns:
        master_df[curr_col] = master_df[curr_col].fillna(master_df[prev_col])

if "VWAP_0to30" in master_df.columns:
    master_df["VWAP_0to30"] = master_df["VWAP_0to30"].fillna(master_df.get("VWAP_0to15", np.nan)).fillna(master_df.get("VWAP_15to30", np.nan))

# 2. Regional Delivery Areas DE1-DE4
all_windows = windows_15m + (["0to30"] if "VWAP_0to30" in master_df.columns else [])
for area in REGIONAL_AREAS:
    for w in all_windows:
        reg_col = f"{area}_VWAP_{w}"
        nat_col = f"VWAP_{w}"
        if reg_col in master_df.columns and nat_col in master_df.columns:
            master_df[reg_col] = master_df[reg_col].fillna(master_df[nat_col])

# 3. Non-price trading features: fill NaN with 0 (no trades / 0 MW / 0 EUR)
non_price_cols = [
    c for c in master_df.columns
    if any(k in c for k in ["total_volume", "total_trades", "id_balance", "trade_intra", "SRL", "MRL"])
]
for c in non_price_cols:
    master_df[c] = master_df[c].fillna(0)

vwap_cols = [c for c in master_df.columns if "VWAP_" in c and not any(x in c for x in ["total_volume", "total_trades", "upper", "lower", "lag_"])]
print(f"Hierarchical imputation completed. Remaining NaNs across all {len(vwap_cols)} VWAP columns: {master_df[vwap_cols].isna().sum().sum()}")

## 8. Multi-Day Lags and Buffer Trimming
Generate d-1 (24h), d-2 (48h), and d-7 (168h) lags for benchmark prices, then trim the 7-day lookback buffer to the delivery horizon (`START_UTC` to `END_UTC`).

In [ ]:
# 8. Multi-Day Lags & Lag Buffer Trimming
lag_shifts = {
    "lag_24h": 96,    # 24 h earlier on the UTC grid (= same local product of d-1 except around clock changes)
    "lag_48h": 192,   # 48 h earlier
    "lag_168h": 672   # 168 h earlier
}

potential_lag_targets = [
    "VWAP_90to105",
    "Dayahead_Auction_hourly",
    "Intraday Auction Price",
    "DE1_VWAP_90to105",
    "DE2_VWAP_90to105",
    "DE3_VWAP_90to105",
    "DE4_VWAP_90to105"
]
cols_to_lag = [c for c in potential_lag_targets if c in master_df.columns]

lagged_dict = {}
for col in cols_to_lag:
    for lag_name, shift_val in lag_shifts.items():
        lagged_dict[f"{col}_{lag_name}"] = master_df[col].shift(shift_val)

lag_df = pd.DataFrame(lagged_dict, index=master_df.index)
master_df = pd.concat([master_df, lag_df], axis=1)

# Trim lag buffer to the canonical delivery horizon [START_UTC, END_UTC]
master_df = master_df[(master_df["Date"] >= START_UTC) & (master_df["Date"] <= END_UTC)].copy()

print(f"Generated {len(lag_df.columns)} multi-day lag features.")
print(f"Trimmed master dataset: {len(master_df):,} quarter-hours ({master_df['Date'].iloc[0]} to {master_df['Date'].iloc[-1]}).")
master_df[["Date", "VWAP_90to105", "VWAP_90to105_lag_24h", "VWAP_90to105_lag_168h"]].head(3)


## 9. Derived Features (S2 neighbor contracts, S4 cross-zonal balances)
Row-wise feature engineering from `feature_engineering.py`; every derived column is registered in `features.csv`.

In [ ]:
# 9. Materialize derived features so that every model input exists in the master dataset
import sys
sys.path.insert(0, os.path.abspath("."))
from feature_engineering import add_derived_features

master_df = add_derived_features(master_df)
print(f"Master dataset with derived features: {master_df.shape}")

## 10. Feature Manifest Alignment
Sort columns according to `features.csv` (incl. the derived features) to ensure consistent ordering across all models.

In [ ]:
# 10. Align Column Ordering via features.csv
if os.path.exists(FEATURES_CSV):
    f_df = pd.read_csv(FEATURES_CSV)
    canonical_order = f_df["name"].dropna().astype(str).str.strip().tolist()

    matched_cols = [c for c in canonical_order if c in master_df.columns]
    unmatched_cols = [c for c in master_df.columns if c not in matched_cols]

    master_df = master_df[matched_cols + unmatched_cols]
    print(f"Aligned with features.csv: {len(matched_cols)} matched, {len(unmatched_cols)} appended.")
else:
    print("features.csv not found; preserving existing order.")

print(f"Master dataset aligned: Rows: {len(master_df):,}, Columns: {len(master_df.columns)}")


## 11. Quality Control
All columns must be complete on the delivery horizon. The ex-post lag buffer before the first delivery may only contain gaps in derived columns (neighbor windows before the start of the data), which the loader drops.

In [ ]:
# 11. Quality Verification & Inspection
targets = [f"{area}_VWAP_0to30" for area in REGIONAL_AREAS]
print("Regional Target Variables Summary Statistics:")
display(master_df[targets].describe().T)

delivery = master_df["Date"] >= pd.Timestamp(FIRST_DELIVERY_UTC, tz="UTC")
derived = set(pd.read_csv(FEATURES_CSV).dropna(subset=["derived_from"])["name"])
total_nans = master_df[delivery].isna().sum().sum()
buffer_nan_cols = master_df.columns[master_df[~delivery].isna().any()]
print(f"\nMissing values on the delivery horizon: {total_nans}")
print(f"Columns with gaps in the lag buffer:   {len(buffer_nan_cols)} (all derived: {set(buffer_nan_cols) <= derived})")
print(f"Timestamp Duplicates:        {master_df['Date'].duplicated().sum()}")
print(f"Total Rows:                  {len(master_df):,}")
print(f"Total Columns:               {len(master_df.columns)}")
print(f"Memory Usage:                {master_df.memory_usage().sum() / (1024 * 1024):.2f} MB")
assert total_nans == 0, f"Error: Dataset contains {total_nans} unexpected NaNs!"
assert set(buffer_nan_cols) <= derived, f"Error: gaps in non-derived buffer columns: {sorted(set(buffer_nan_cols) - derived)}"
non_num_cols = [c for c in master_df.columns if c != "Date" and not pd.api.types.is_numeric_dtype(master_df[c])]
assert len(non_num_cols) == 0, f"Error: Dataset contains non-numeric columns: {non_num_cols}!"

## 12. Parquet Export
Save the finalized master dataset to `Data/master_dataset.parquet`.

In [ ]:
# 12. Export as Snappy-compressed Parquet for high-speed I/O
output_parquet = os.path.join(DATA_DIR, "master_dataset.parquet")
print(f"Exporting master dataset to Parquet: {output_parquet} ...")
t_p0 = time.time()

# Ensure Date is properly timezone-aware (UTC) directly without RAM copy
if master_df["Date"].dt.tz is None:
    master_df["Date"] = master_df["Date"].dt.tz_localize("UTC")

master_df.to_parquet(output_parquet, engine="pyarrow", compression="snappy", index=False)
parquet_size_mb = os.path.getsize(output_parquet) / (1024 * 1024)
print(f"Parquet export complete in {time.time()-t_p0:.1f}s! Size: {parquet_size_mb:.2f} MB")
